# Separate features for each axes (X, Y, Z)

In [1]:
import nibabel as nib
import numpy as np
import pandas as pd

def compute_axis_shape_features(mask_nii_path, min_voxels=10):
    nii = nib.load(mask_nii_path)
    mask = nii.get_fdata().astype(int)
    affine = nii.affine

    lesion_labels = np.unique(mask)
    lesion_labels = lesion_labels[lesion_labels != 0]

    all_lesions = []
    for label in lesion_labels:
        coords_voxel = np.column_stack(np.where(mask == label))

        if coords_voxel.shape[0] < min_voxels:
            continue  # skip tiny lesions

        # ===== voxel → world (mm) =====
        coords_h = np.c_[coords_voxel, np.ones(coords_voxel.shape[0])]
        coords_mm = (affine @ coords_h.T).T[:, :3]

        # ===== center =====
        coords_mm -= coords_mm.mean(axis=0)
        var_x, var_y, var_z = np.var(coords_mm, axis=0)
        eps = 1e-8

        lesion_features = {
            "lesion_label": int(label),

            # elongation (axis-wise)
            "elong_x": var_x / (var_y + var_z + eps),
            "elong_y": var_y / (var_x + var_z + eps),
            "elong_z": var_z / (var_x + var_y + eps),

            # flatness (axis-wise)
            "flat_x": var_x / (max(var_y, var_z) + eps),
            "flat_y": var_y / (max(var_x, var_z) + eps),
            "flat_z": var_z / (max(var_x, var_y) + eps),
        }

        all_lesions.append(lesion_features)

    return pd.DataFrame(all_lesions)

In [2]:
mask_path = r"C:\Users\lukas\OneDrive - VUT\Plocha\Diploma thesis\Myel_001\Myel_001_lesions_seg_validation_VV_final.nii.gz"

df_lesions = compute_axis_shape_features(mask_path)
print(df_lesions.head(3))

   lesion_label   elong_x   elong_y   elong_z    flat_x    flat_y    flat_z
0          1004  0.191145  0.209643  1.995973  0.240870  0.260139  3.844096
1          1015  0.320543  0.949029  0.370502  0.498509  1.801152  0.555200
2          1024  0.226970  2.009636  0.172721  0.277032  3.609687  0.220570


# úprava tvarov: uzavrenie lézí

In [3]:
import nibabel as nib
import numpy as np
from scipy.ndimage import binary_closing, binary_opening, binary_fill_holes
from skimage.morphology import ball


def morphologically_refine_lesions(mask_nii_path, output_nii_path, operation="closing", radius=1, min_voxels=10):

    nii = nib.load(mask_nii_path)
    mask = nii.get_fdata().astype(int)
    affine = nii.affine
    header = nii.header

    refined_mask = np.zeros_like(mask)
    labels = np.unique(mask)
    labels = labels[labels != 0]

    selem = ball(radius)

    for label in labels:
        lesion = mask == label

        if lesion.sum() < min_voxels:
            refined_mask[lesion] = label
            continue

        if operation == "closing":
            refined = binary_closing(lesion, structure=selem)

        elif operation == "opening":
            refined = binary_opening(lesion, structure=selem)

        elif operation == "fill_holes":
            refined = binary_fill_holes(lesion)

        else:
            raise ValueError("Unknown operation")

        refined_mask[refined] = label

    refined_nii = nib.Nifti1Image(refined_mask.astype(np.int16), affine, header)
    nib.save(refined_nii, output_nii_path)
    print("Closed mask successfully saved.")

In [4]:
morphologically_refine_lesions(
    mask_nii_path=r"C:\Users\lukas\OneDrive - VUT\Plocha\Diploma thesis\Myel_001\Myel_001_lesions_seg_validation_VV_final.nii.gz",
    output_nii_path=r"C:\Users\lukas\OneDrive - VUT\Plocha\Diploma thesis\Myel_001\Myel_001_lesions_seg_closed.nii.gz"
)

Closed mask successfully saved.
